# 3장 · 엔트로피 — 불확실성을 재는 자 🧭

웹 챕터 **"엔트로피"** 와 짝을 이루는 Python 실습입니다.
웹의 탐험대에서 잡은 감(균등 최대·몰빵 0)을, 여기서는 **직접 계산·무작위 검증·실전 텍스트**로 굳힙니다.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch3')

## 1. 엔트로피 함수: 이 과정 내내 쓸 4줄

$H = \sum p \cdot \log_2 \frac{1}{p}$ 를 그대로 코드로 옮깁니다.
확률 0인 항은 **0·log 0 = 0 관례**대로 건너뜁니다 (안 일어나는 일은 평균에 낄 자격이 없음).

In [ ]:
import numpy as np

def entropy(p):
    """분포 p 의 엔트로피 (비트). 0·log0 = 0 관례로 p=0 항은 제외."""
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(np.sum(p * np.log2(1 / p)))

print(f'공정 동전 (1/2, 1/2)          : {entropy([1/2, 1/2]):.3f} 비트')
print(f'치우친 동전 (0.9, 0.1)        : {entropy([0.9, 0.1]):.3f} 비트')
print(f'몰빵 (1, 0, 0, 0)             : {entropy([1, 0, 0, 0]):.3f} 비트')
print(f'4면 균등 (1/4 × 4)            : {entropy([1/4] * 4):.3f} 비트  (= log₂4)')
print(f'주사위 6면 균등               : {entropy([1/6] * 6):.3f} 비트  (= log₂6 = {np.log2(6):.3f})')
print(f'날씨 (1/2, 1/4, 1/8, 1/8)     : {entropy([1/2, 1/4, 1/8, 1/8]):.3f} 비트  (2장의 1.75!)')

## 2. 동전 엔트로피 곡선: 역U 그리기

앞면 확률 p 를 0에서 1까지 움직이며 H 를 그립니다. **p = 0.5 에서 최대 1비트**,
양 끝(결과가 확실)에서 0. 엔트로피의 상징인 역U 곡선입니다.

In [ ]:
import matplotlib.pyplot as plt

# 한글 폰트 설정 (안 하면 한글 제목/라벨이 네모로 깨짐)
# Windows: 'Malgun Gothic' / macOS: 'AppleGothic' / Linux: 'NanumGothic' 로 바꾸세요
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False  # 음수 부호(-) 깨짐 방지

ps = np.linspace(0.001, 0.999, 400)
Hs = [entropy([p, 1 - p]) for p in ps]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ps, Hs, color='#6366f1', lw=2.5)
ax.plot(0.5, 1.0, 'o', color='#4f46e5', ms=9)
ax.annotate('p=0.5 에서 최대 1비트\n(가장 못 맞히는 반반)', (0.5, 1.0),
            textcoords='offset points', xytext=(10, -30), fontsize=10)
ax.set_xlabel('앞면이 나올 확률 p')
ax.set_ylabel('엔트로피 H (비트)')
ax.set_title('동전의 엔트로피: 확실할수록(양 끝) 불확실성이 죽는다')
ax.grid(True, color='#eef2f7')
plt.show()

## 3. "균등이 최대"를 이길 수 있을까: 무작위 도전 10,000번

경우가 4개인 분포를 무작위로 10,000개 만들어 H 를 재 봅니다.
단 하나라도 log₂4 = 2비트를 넘으면 교과서를 다시 써야 합니다. 과연?

In [ ]:
rng = np.random.default_rng(42)

# 무작위 분포 10,000개 (양수 4개를 뽑아 합이 1이 되게 정규화)
raw = rng.random((10000, 4)) + 1e-12
dists = raw / raw.sum(axis=1, keepdims=True)
Hs = np.array([entropy(d) for d in dists])

print(f'10,000개 중 최대 H : {Hs.max():.6f} 비트')
print(f'이론적 최대 log₂4  : {np.log2(4):.6f} 비트')
assert Hs.max() <= np.log2(4) + 1e-9, '균등을 이기는 분포가 나왔다?! (그럴 수 없음)'

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(Hs, bins=60, color='#6366f1', alpha=0.85)
ax.axvline(np.log2(4), color='#ef4444', lw=2, linestyle='--', label='이론적 최대 log₂4 = 2비트')
ax.set_xlabel('엔트로피 H (비트)')
ax.set_ylabel('분포 개수')
ax.set_title('무작위 분포 10,000개의 H: 아무도 빨간 선(균등)을 넘지 못한다')
ax.legend()
plt.show()

## 4. 실전: 텍스트의 엔트로피 (2장과의 재회)

2장에서 허프만으로 압축했던 **같은 영어 문장**의 글자 분포 엔트로피를 재 봅니다.
그때 허프만의 평균이 글자당 4.05비트였죠. 바닥(H)은 얼마였을까요?

In [ ]:
from collections import Counter

text = ('information theory studies the quantification storage and communication '
        'of information the field was established by claude shannon in the forties')

counts = np.array(list(Counter(text).values()), dtype=float)
p_letters = counts / counts.sum()
H_text = entropy(p_letters)

print(f'글자 종류        : {len(counts)}가지')
print(f'균등이라면       : log₂{len(counts)} = {np.log2(len(counts)):.3f} 비트')
print(f'실제 분포의 H    : {H_text:.3f} 비트   ← 압축의 바닥')
print(f'2장 허프만 평균  : 4.048 비트   ← 바닥 바로 위!')
print(f'ASCII 고정 길이  : 8.000 비트')
print('\n글자가 균등하지 않은(치우친) 덕분에, 8비트짜리 글자를 4비트 남짓으로 보낼 수 있는 거예요.')

## 5. AI 미리보기: 두 모델의 "자신 없음" 재기

같은 문맥에서 두 모델이 내놓은 다음 토큰 분포입니다. 누가 더 갈팡질팡하고 있을까요?

In [ ]:
tokens = ['서울', '부산', '평양', '한강', '김치']
confident = np.array([0.94, 0.02, 0.02, 0.01, 0.01])   # "대한민국의 수도는" 다음 — 확신
unsure = np.array([0.30, 0.25, 0.20, 0.15, 0.10])      # 애매한 문맥 — 갈팡질팡

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, dist, name in [(axes[0], confident, '확신 모델'), (axes[1], unsure, '갈팡질팡 모델')]:
    ax.bar(tokens, dist, color='#6366f1', alpha=0.85)
    ax.set_title(f'{name}: H = {entropy(dist):.2f} 비트')
    ax.set_ylim(0, 1)
axes[0].set_ylabel('다음 토큰 확률')
fig.suptitle('분포의 엔트로피 = 모델의 자신 없음 (8장 temperature 가 이 값을 돌리는 다이얼)')
plt.tight_layout()
plt.show()

print(f'확신 모델     H = {entropy(confident):.3f} 비트 (몰빵 → 낮음)')
print(f'갈팡질팡 모델 H = {entropy(unsure):.3f} 비트 (퍼짐 → 균등 log₂5 = {np.log2(5):.3f} 에 근접)')

## 6. 직접 풀어보기 ✏️

"내일 날씨" 분포가 맑음 1/2 · 흐림 1/4 · 비 1/8 · 눈 1/8 일 때,
엔트로피는 몇 비트일까요? (웹 챕터의 어려움 문제와 같은 계산)

In [ ]:
# TODO: 아래 answer 를 여러분의 계산으로 바꾸고 실행하세요.
# 힌트: 항별로 p × log₂(1/p) — 또는 entropy([1/2, 1/4, 1/8, 1/8])

answer = entropy([1/2, 1/4, 1/8, 1/8])
print('내 답:', answer)
assert abs(answer - 1.75) < 1e-9, '다시 계산해 보세요 — 0.5 + 0.5 + 0.375 + 0.375'
print('✅ 잘했어요! 평균 놀람 = 압축의 바닥 = 평균 질문 수, 전부 1.75비트입니다.')

---
**오늘의 한 줄**: 엔트로피 H = 평균 놀람. 몰빵이면 0, 균등이면 최대 log₂n,
그리고 무손실 압축이 절대 못 뚫는 바닥.

다음 챕터는 이 과정의 핵심인 **틀린 지도(q)로 세상(p)을 읽는 비용, 크로스 엔트로피**입니다.
LLM 손실함수의 정체가 드디어 밝혀집니다. 🙂